In [ ]:
import pandas as pd
import numpy as np
import json
import re

In [ ]:
listing_data = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\data\raw\listings.csv.gz")

In [ ]:
print(listing_data.head().to_string())

In [ ]:
listing_data.shape

In [ ]:
listing_data.info()

In [ ]:
listing_data.describe()

In [ ]:
listing_data.duplicated().sum()

In [ ]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

In [ ]:
# Summary of the listing dataset

# - The DataFrame has 36445 rows and 85 columns
# - The data dont have any duplicate value, but it has Nan values.

# - Change datatypes = {
#                       has_availability (str <-> Boolean)
#                       host_has_profile_pic (str <-> Boolean)
#                       host_identity_verified (str <-> Boolean)
#                       host_is_superhost (str <-> Boolean)
#
#                       first_review (str <-> datetime)
#                       bathrooms_text (str <-> float)
#                       amenities (list <-> Json-like string)
#                       }

# - As we have to prepare the data to train a model that predicts Nightly price of a property
#   Therefore, we have to drop few columns

In [ ]:
#Dropping columns
cols_to_drop = [
    # 100% null
    'neighborhood_overview', 'host_since', 'host_response_time',
    'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url',
    'host_neighbourhood', 'host_total_listings_count', 'host_verifications',
    'neighbourhood', 'calendar_updated', 'estimated_revenue_l365d',
    'instant_bookable',
    # URLs and IDs
    'listing_url', 'scrape_id', 'last_scraped', 'picture_url',
    'host_url', 'host_profile_id', 'host_profile_url',
    'host_picture_url', 'calendar_last_scraped',
    # free text
    'name', 'description', 'host_about', 'host_name',
    # high null + not recoverable
    'license', 'host_location',
]

listing_data = listing_data.drop(columns=cols_to_drop)

listing_data = listing_data.drop(columns=['beds'])

listing_data = listing_data.drop(columns=[
    'minimum_minimum_nights', 'maximum_minimum_nights',
    'minimum_maximum_nights', 'maximum_maximum_nights'
])

listing_data = listing_data.drop(columns=['source'])
print(listing_data.shape)


In [ ]:
listing_data['bathrooms_text'].value_counts().head(20)

In [ ]:
#Changing Datatypes
bool_cols = ['has_availability', 'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']

for col in bool_cols:
    listing_data[col] = listing_data[col].str.strip().str.lower().map({'t': True, 'f': False})

listing_data['first_review'] = pd.to_datetime(listing_data['first_review'])
listing_data['last_review'] = pd.to_datetime(listing_data['last_review'])

#converting bathroom_text to float values
def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text).lower().strip()
    
    # half bath = 0.5
    if 'half' in text:
        return 0.5
    
    # extract the first number (handles 1, 1.5, 2 etc.)
    match = re.search(r'[\d.]+', text)
    if match:
        return float(match.group())
    
    return np.nan

listing_data['bathrooms_parsed'] = listing_data['bathrooms_text'].apply(parse_bathrooms)
listing_data = listing_data.drop(columns=['bathrooms'])
listing_data = listing_data.rename(columns={'bathrooms_parsed': 'bathrooms'})
listing_data = listing_data.drop(columns=['bathrooms_text'])
#placing bathroom_prased before bedrooms
cols = list(listing_data.columns)
cols.remove('bathrooms')
bedrooms_idx = cols.index('bedrooms')
cols.insert(bedrooms_idx, 'bathrooms')
listing_data = listing_data[cols]


In [ ]:
listing_data = listing_data.drop(columns=['host_has_profile_pic', 'host_identity_verified'])

In [ ]:
listing_data = listing_data.drop(columns=[
    'price_quote_checkin_date',
    'price_quote_checkout_date', 
    'price_quote_total_price',
    'price_quote_price_per_night',
    'price_quote_raw'
])

In [ ]:
listing_data['price'] = listing_data['price'].str.replace('[$,]', '', regex=True).astype(float)

In [ ]:
listing_data['price'].describe()

In [ ]:
print(listing_data.head(10).to_string())

In [ ]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

In [ ]:
# after seeing the null % summary again, we can see that only a few columns has significant null %
# therefore, for price - we have to drop the rows, as 40% is very high
#                bedrooms - fill it with median grouped by property_type
#                bathrooms -  fill it with median grouped by room_type
#                for all the host_time_as_user/host_years/month - fill it with median ()
#                for all the host_is_superhost - fill it with mode
#                has_availability - fill it with mode
#                first and last review - we can modify it to days_since_reviewed and fill the nan values with a large number
#                review scores_* -can't fill it with zero (as it means terrible listing), so will create has_reviews flag to preserve previous value and then fill null values with median
#                reviews_per_month - fill it with zero

In [ ]:
#filling null values

listing_data = listing_data.dropna(subset=['price'])

host_numeric_column = ['hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_listings_count']
for col in host_numeric_column:
    listing_data[col] = listing_data[col].fillna(listing_data[col].median())

listing_data['host_is_superhost'] = listing_data['host_is_superhost'].fillna(listing_data['host_is_superhost'].mode()[0])

listing_data['bathrooms'] = listing_data.groupby('room_type')['bathrooms'].transform(lambda x: x.fillna(x.median()))

listing_data['bedrooms'] = listing_data.groupby('property_type')['bedrooms'].transform(lambda x: x.fillna(x.median()))

listing_data['has_availability'] = listing_data['has_availability'].fillna(listing_data['has_availability'].mode()[0])

listing_data['reviews_per_month'] = listing_data['reviews_per_month'].fillna(0)

listing_data['has_reviews'] = listing_data['review_scores_rating'].notna().astype(int)
cols = list(listing_data.columns)
cols.remove('has_reviews')
rating_idx = cols.index('review_scores_rating')
cols.insert(rating_idx, 'has_reviews')
listing_data = listing_data[cols]

review_cols = ['review_scores_rating', 'review_scores_accuracy', 
               'review_scores_cleanliness', 'review_scores_checkin',
               'review_scores_communication', 'review_scores_location', 
               'review_scores_value']

for col in review_cols:
    listing_data[col] = listing_data[col].fillna(listing_data[col].median())

today = pd.Timestamp.today()

listing_data['days_since_first_review'] = (today - listing_data['first_review']).dt.days
listing_data['days_since_last_review'] = (today - listing_data['last_review']).dt.days

listing_data['days_since_first_review'] = listing_data['days_since_first_review'].fillna(99999)
listing_data['days_since_last_review'] = listing_data['days_since_last_review'].fillna(99999)

cols = list(listing_data.columns)
cols.remove('days_since_first_review')
cols.remove('days_since_last_review')
first_idx = cols.index('first_review')
cols.insert(first_idx, 'days_since_first_review')
cols.insert(first_idx + 1, 'days_since_last_review')
listing_data = listing_data[cols]

listing_data = listing_data.drop(columns=['first_review', 'last_review'])



In [ ]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

In [ ]:
#removing tiny remaining nulls from few columns 
listing_data = listing_data.dropna(subset=['bedrooms', 'minimum_nights', 'has_availability'])

In [ ]:
Null_percent = pd.DataFrame({
    'missing_percent': listing_data.isnull().mean() * 100
})

print(Null_percent.to_string())

In [ ]:
listing_data.shape

In [ ]:
#modifying the amenities 
print(type(listing_data['amenities'][0]))
print(listing_data['amenities'][0])

In [ ]:
#prasing the whole column to actual lists
listing_data['amenities'] = listing_data['amenities'].apply(json.loads)

listing_data['amenity_count'] = listing_data['amenities'].apply(len)

In [ ]:
listing_data['amenities'].explode().value_counts().head(20)

In [ ]:
amen_counts = listing_data['amenities'].explode().value_counts()
wanted = ['Pool', 'Gym', 'Elevator', 'Parking', 'Washer', 'Dishwasher']
amen_counts.reindex(wanted, fill_value=0).sort_values(ascending=False)

In [ ]:
top_amenities = ['Wifi', 'Kitchen', 'Air conditioning', 
                 'Dedicated workspace', 'TV', 'Heating', 
                 'Refrigerator', 'Microwave', 'Washer', 
                 'Dishwasher', 'Elevator', 'Gym']

for amenity in top_amenities:
    col_name = f"has_{amenity.lower().replace(' ', '_')}"
    listing_data[col_name] = listing_data['amenities'].apply(lambda x: 1 if amenity in x else 0)

In [ ]:
has_cols = [c for c in listing_data.columns if c.startswith('has_')]
print(has_cols)
listing_data[has_cols].head()

In [ ]:
listing_data = listing_data.drop(columns=['amenities'])

In [ ]:
#saving the cleaned file

listing_data.to_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\data\processed\listing_clean.csv', index=False)
print(listing_data.shape)